# 04 Portfolio Walk-forward

Notebook này kiểm tra độ ổn định OOS của portfolio qua nhiều cửa sổ thời gian.

Điểm cần đọc chính:
- Có bao nhiêu window dương?
- Window xấu nhất lỗ bao nhiêu?
- Drawdown OOS có ổn định không?
- Sharpe/PF có chỉ tốt ở một vài window hay giữ được qua nhiều giai đoạn?


In [ ]:
# Cell 1 - Bootstrap đường dẫn import an toàn
#
# Notebook có thể được mở từ repo root, từ thư mục strategies/combo,
# hoặc từ một working directory khác trong VS Code/Jupyter. Vì vậy ta không
# dùng `config.py` làm marker root: trong strategies/combo cũng có config.py,
# rất dễ nhận nhầm thư mục strategy là repo root.
#
# Marker đáng tin cậy hơn là pyproject.toml + thư mục core_python/shared.
# Sau khi tìm được root thật, ta thêm cả repo root và core_python vào sys.path
# để import được `shared.*` và `strategies.combo.*`.

import sys
from pathlib import Path


def _find_root(start: Path, marker: str = 'pyproject.toml') -> Path:
    for p in [start, *start.parents]:
        if (p / marker).exists() and (p / 'core_python' / 'shared').exists():
            return p
    raise RuntimeError(f'Không tìm thấy repo root chứa {marker!r} và core_python/shared')


ROOT = _find_root(Path.cwd())
CORE = ROOT / 'core_python'
for p in (str(ROOT), str(CORE)):
    if p not in sys.path:
        sys.path.insert(0, p)

print('ROOT =', ROOT)
print('CORE =', CORE)


In [ ]:
# Cell 2 - Import thư viện và helper walk-forward

from IPython.display import display

from core_python.strategies.combo.params import summary as strategy_summary
from core_python.strategies.combo.research_utils import (
    configure_notebook,
    plot_walkforward_dashboard,
    show_note,
    show_run_config,
    summarize_walkforward,
)
from core_python.strategies.combo.portfolio.walkforward import walk_forward_portfolio

configure_notebook()
print(strategy_summary())


In [ ]:
# Cell 3 - Cấu hình walk-forward
#
# SYMBOL_PARAMS là bộ tham số đã chọn cho từng symbol.
# IS_BARS: số bar in-sample dùng để tối ưu/chọn tham số.
# OOS_BARS: số bar out-of-sample dùng để đánh giá thật.
# STEP_BARS: khoảng dịch cửa sổ. STEP nhỏ hơn OOS sẽ tạo window chồng lấn.

SYMBOL_PARAMS = {
    'US30':  {'x': 10.0, 'ktp': 2.3, 'ma_period': 20, 'trailing_activation': 1.0},
    'US500': {'x': 1.0,  'ktp': 2.3, 'ma_period': 20, 'trailing_activation': 1.0},
    'DE40':  {'x': 5.0,  'ktp': 2.3, 'ma_period': 20, 'trailing_activation': 1.0},
    'GOLD':  {'x': 0.5,  'ktp': 2.3, 'ma_period': 20, 'trailing_activation': 1.0},
}

RUN_CONFIG = {
    'account_mode': 'standard',
    'initial_balance': 100_000.0,
    'is_bars': 5_000,
    'oos_bars': 1_250,
    'step_bars': 1_250,
    'max_bars': 40_000,
}

show_run_config('Cấu hình portfolio walk-forward', RUN_CONFIG)
show_note('Symbol params', 'Các tham số dưới đây là bộ đang được kiểm định qua OOS windows.')
display(SYMBOL_PARAMS)


In [ ]:
# Cell 4 - Chạy walk-forward portfolio

wf_df, wf_summary = walk_forward_portfolio(
    SYMBOL_PARAMS,
    account_mode=RUN_CONFIG['account_mode'],
    initial_balance=RUN_CONFIG['initial_balance'],
    is_bars=RUN_CONFIG['is_bars'],
    oos_bars=RUN_CONFIG['oos_bars'],
    step_bars=RUN_CONFIG['step_bars'],
    max_bars=RUN_CONFIG['max_bars'],
)

show_note('Walk-forward summary raw', 'Summary gốc từ runner, giữ lại để đối chiếu.')
print(wf_summary)
display(wf_df)


In [ ]:
# Cell 5 - Stability dashboard
#
# Đây là phần quan trọng nhất của walk-forward: xem sự ổn định qua từng OOS window.
# Một chiến lược tốt nên có nhiều window dương, drawdown không bùng lên ở một vài giai đoạn.

wf_stability = summarize_walkforward(wf_df)
plot_walkforward_dashboard(wf_df)

metric_cols = [
    c for c in ['window', 'oos_start', 'oos_end', 'total_return', 'max_drawdown', 'sharpe', 'profit_factor', 'win_rate']
    if c in wf_df.columns
]
if metric_cols:
    display(wf_df[metric_cols])


In [ ]:
# Cell 6 - Export thủ công nếu cần
#
# Walk-forward trả DataFrame riêng, nên export trực tiếp bằng pandas khi cần.

EXPORT_REPORT = False
if EXPORT_REPORT and not wf_df.empty:
    out_dir = ROOT / 'reports' / 'combo' / 'portfolio_walkforward'
    out_dir.mkdir(parents=True, exist_ok=True)
    wf_df.to_csv(out_dir / 'walkforward_windows.csv', index=False, encoding='utf-8-sig')
    wf_stability.to_csv(out_dir / 'walkforward_summary.csv', encoding='utf-8-sig')
    print('Đã export vào:', out_dir)
else:
    print('Export đang tắt. Đổi EXPORT_REPORT = True nếu muốn lưu CSV.')
